<a href="https://colab.research.google.com/github/KoteswaraRaoSurimalli/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:

from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float


In [4]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [14]:
# pip install python-docx

from docx import Document
import json
import re


def show_questions_as_json(docx_file):
    """
    Read all questions from DOCX
    and display them in JSON format.
    """

    # Load DOCX file
    doc = Document(docx_file)

    questions = []

    # Read paragraphs
    for para in doc.paragraphs:
        text = para.text.strip()

        # Skip empty lines
        if not text:
            continue

        # Remove numbering
        cleaned_text = re.sub(
            r'^(Q\s*\d+[\.\)]?|\d+[\.\)])\s*',
            '',
            text,
            flags=re.IGNORECASE
        )

        # Add to list
        questions.append({
            "id": len(questions) + 1,
            "question": cleaned_text
        })

    # Convert to JSON
    json_output = json.dumps(
        {"questions": questions},
        indent=4,
        ensure_ascii=False
    )

    # Show JSON
    print(json_output)


# =========================
# Example Usage
# =========================

docx_file = "/content/sample_data/Java_MCQ.docx"

show_questions_as_json(docx_file)

{
    "questions": [
        {
            "id": 1,
            "question": "Java Programming"
        },
        {
            "id": 2,
            "question": "Multiple Choice Questions"
        },
        {
            "id": 3,
            "question": "Which of the following is the correct way to declare a variable in Java?"
        },
        {
            "id": 4,
            "question": "✔ A) int x = 10;"
        },
        {
            "id": 5,
            "question": "B) x int = 10;"
        },
        {
            "id": 6,
            "question": "C) var x := 10;"
        },
        {
            "id": 7,
            "question": "D) integer x = 10;"
        },
        {
            "id": 8,
            "question": "Explanation: In Java, variable declarations follow the syntax: datatype variableName = value;"
        },
        {
            "id": 9,
            "question": "What is the default value of a boolean variable in Java?"
        },
        {
            "id": 10,
 